# Step 2 — Across-trial oscillation summary (mean ± SEM)

**Input contract** (produced by Step 1 Block 6, see `step1_phantom_video_analysis.ipynb`):
- every `<stem>_combined.csv` under `DATA_DIR` on a uniform 20 kHz grid in absolute NIDAQ seconds
- required columns: `time_s`, `rescaled_intensity`, `wingbeat_amplitude`, `wingbeat_frequency`, `inferred_flight_power`

**What this notebook does**
1. Loads every combined CSV in `DATA_DIR`.
2. Clips each trial to absolute time window `[ABS_T0, ABS_T1]` and interpolates to a common grid.
3. Computes mean ± SEM across trials for 4 signals: wingbeat amplitude, wingbeat frequency, inferred flight power, spiracle aperture.
4. Saves a single summary CSV (grid × [mean, sem, n] per metric) + a 4-panel SVG/PNG figure.

Window and output names are controlled by `ABS_T0`, `ABS_T1`, and `OUT_PREFIX` in Block 1.

---

## 中文说明

**用途**：分析流程第二步的"振荡视觉刺激"汇总版本。它把同一条件下多次实验（trial）的行为信号对齐到同一时间网格，计算跨实验的均值 ± 标准误（mean ± SEM），并进一步做"周期折叠"平均（把重复的振荡周期叠在一起）和相关性分析。用于刻画在 0.05 Hz 振荡视觉刺激下，气门开度与飞行参数随刺激起伏的群体平均反应。

**预期输入**：
- 文件类型：`DATA_DIR` 下的每个 `<stem>_combined.csv`（Step 1 输出，均匀 20 kHz 网格，时间轴为 NIDAQ 绝对秒）。这里只关注 `_combined.csv` 这个后缀约定，基础名不限。
- 必需列：`time_s`（时间，秒）、`rescaled_intensity`（气门开度）、`wingbeat_amplitude`（翅拍幅度，rad）、`wingbeat_frequency`（翅拍频率，Hz）、`inferred_flight_power`（推算飞行功率，W/kg，由翅拍幅度与频率推算）。

**预期输出**（均写入 `DATA_DIR`）：
- `oscillation_summary_<T0>_<T1>s.csv` 及配套 `.svg` / `.png`：全窗口的跨实验 mean ± SEM 曲线。
- `oscillation_cycle_avg_0_20s.csv` 及配套图：把多个 20 s 周期折叠平均后的单周期曲线（Block 9）。
- `pearson_correlations_<T0>_<T1>s.csv`：每次实验、每个信号对的皮尔逊相关系数（Block 10）。
- `pearson_corr_20_40s_vs_60_80s*`（图 + 配对统计 CSV）：两个时间窗相关系数的配对比较（Block 10b）。
- `spiracle_peak_trough_comparison_20_40_vs_60_80.csv`：气门峰/谷/摆幅的两窗配对比较（Block 11）。

分析窗口与输出文件名由 Block 1 中的 `ABS_T0`、`ABS_T1`、`OUT_PREFIX` 控制。

### Block 1 — Imports & parameters

Sets the run parameters: `DATA_DIR` (folder of combined CSVs), the absolute time window to summarize (`ABS_T0`, `ABS_T1`), the resampling grid resolution (`N_GRID`, ≈100 Hz — far above the ~0.05 Hz oscillation), and the window-aware output filename prefix `OUT_PREFIX`.

**中文**：设置运行参数：`DATA_DIR`（合并 CSV 所在文件夹）、要汇总的绝对时间窗 `[ABS_T0, ABS_T1]`、重采样网格点数 `N_GRID`（约 100 Hz，远高于 ~0.05 Hz 的振荡频率以保证细节）、以及带时间窗信息的输出文件名前缀 `OUT_PREFIX`。

In [47]:
# Block 1 — imports + parameters
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Folder to scan for Step 1 outputs
DATA_DIR  = Path(r'C:\Users\Lylah\Desktop\data_processing\oscillating')
RECURSIVE = False

# Absolute NIDAQ time window to summarize
ABS_T0 = 20.0
ABS_T1 = 80.0

# Resampling grid: 100 Hz effective (6000 samples / 60 s) — far above the ~0.05 Hz oscillation
N_GRID = 6000

# Output filename prefix (window-aware)
OUT_PREFIX = f"oscillation_summary_{int(ABS_T0)}_{int(ABS_T1)}s"

### Block 2 — Find & load combined CSVs

`find_combined_csvs` returns a sorted list of every `*_combined.csv` in `DATA_DIR`; `load_combined_csv` reads one and verifies the required columns are present.

**中文**：`find_combined_csvs` 返回 `DATA_DIR` 中所有 `*_combined.csv` 的有序列表；`load_combined_csv` 读取单个文件并检查必需列是否齐全。

In [48]:
# Block 2 — find + load combined CSVs

REQUIRED_COLS = [
    "time_s",
    "rescaled_intensity",
    "wingbeat_amplitude",
    "wingbeat_frequency",
    "inferred_flight_power",
]


def find_combined_csvs(data_dir, recursive=False):
    """Return a sorted list of every <stem>_combined.csv in data_dir."""
    data_dir = Path(data_dir)
    it = data_dir.rglob("*_combined.csv") if recursive else data_dir.glob("*_combined.csv")
    return sorted(it)


def load_combined_csv(csv_path):
    df = pd.read_csv(csv_path)
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} missing columns: {missing}")
    return df

### Block 3 — Interpolation helper

`interp_irregular` resamples an irregular `(t, y)` signal onto an arbitrary query grid. It is NaN-safe (drops non-finite samples), sorts by time, removes duplicate timestamps, and returns NaN outside the data's time range rather than extrapolating. This is the same pattern used in Step 1, so every trial can be placed on one common grid.

**中文**：`interp_irregular` 把不规则的 `(t, y)` 信号重采样到任意查询网格上。它对 NaN 安全（先剔除非有限值）、按时间排序、去除重复时间戳，并在数据时间范围之外返回 NaN（而非外推）。与 Step 1 使用同一套做法，从而把每次实验对齐到同一公共网格。

In [49]:
# Block 3 — interp helper (Step 1 Block 6 pattern: NaN-safe, out-of-bounds -> NaN)

def interp_irregular(t, y, xq):
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(t) & np.isfinite(y)
    t = t[m]
    y = y[m]
    if t.size == 0:
        return np.full(xq.shape, np.nan)
    order = np.argsort(t)
    t = t[order]
    y = y[order]
    uniq = np.concatenate([[True], np.diff(t) > 1e-15])
    t = t[uniq]
    y = y[uniq]
    if t.size < 2:
        return np.full(xq.shape, np.nan)
    return np.interp(xq, t, y, left=np.nan, right=np.nan)

### Block 4 — Stack trials onto the common grid

Defines the plot/column order (`METRICS`) and per-metric y-axis limits (`YLIMS`, shared by Blocks 6 and 9). `stack_trials` clips each trial to `[ABS_T0, ABS_T1]`, interpolates every signal onto one shared time grid, and stacks them into arrays of shape `(n_trials, n_grid)` — one per signal — ready for averaging.

**中文**：定义绘图/列的顺序（`METRICS`）以及每个指标的 y 轴范围（`YLIMS`，Block 6 与 Block 9 共用）。`stack_trials` 把每次实验裁剪到 `[ABS_T0, ABS_T1]`，将每个信号插值到同一公共时间网格上，并堆叠成形如 `(实验数, 网格点数)` 的数组（每个信号一个），供后续求平均。

In [50]:
# Block 4 — stack each trial onto the common grid

# Plot/column order: amplitude, frequency, power, aperture (as requested)
METRICS = [
    ("wingbeat_amplitude",    "Wingbeat amplitude",    "rad"),
    ("wingbeat_frequency",    "Wingbeat frequency",    "Hz"),
    ("inferred_flight_power", "Inferred flight power", "W/kg"),
    ("rescaled_intensity",    "Spiracle aperture",     ""),
]

# y-axis limits per metric. Shared by plot_oscillation_summary (Block 6) AND
# plot_cycle_summary (Block 9) so both figures use the same scale. Set to None
# (omit the key) to autoscale that panel. A clipping warning is printed at
# render time when the mean+/-SEM band falls outside the configured range.
YLIMS = {
    "rescaled_intensity":    (0.0, 1.0),
    "wingbeat_amplitude":    (3.14, 6.28),
    "wingbeat_frequency":    (100.0, 300.0),
    "inferred_flight_power": (100.0, 400.0),
}


def stack_trials(csv_paths, abs_t0, abs_t1, n_grid):
    """Return (t_grid, stacks_dict, trial_labels). Each stack has shape (n_trials, n_grid)."""
    t_grid = np.linspace(abs_t0, abs_t1, n_grid)
    stacks = {col: [] for col, *_ in METRICS}
    trial_labels = []

    for p in csv_paths:
        try:
            df = load_combined_csv(p)
        except Exception as e:
            print(f"[warn] {p.name}: {e} — skipping")
            continue

        mask = (df["time_s"] >= abs_t0) & (df["time_s"] <= abs_t1)
        sub = df.loc[mask]
        if len(sub) == 0:
            print(f"[warn] {p.name}: no samples in [{abs_t0}, {abs_t1}] s — skipping")
            continue

        t = sub["time_s"].values
        for col, *_ in METRICS:
            stacks[col].append(interp_irregular(t, sub[col].values, t_grid))
        trial_labels.append(p.stem)

    stacks = {
        k: (np.asarray(v) if len(v) > 0 else np.empty((0, n_grid)))
        for k, v in stacks.items()
    }
    return t_grid, stacks, trial_labels

### Block 5 — Mean ± SEM

`mean_sem` collapses a `(n_trials, n_grid)` stack to per-grid-point mean, SEM and effective n. It is NaN-aware: the mean and SD ignore NaNs, and n counts only finite samples at each time point, so partially missing trials still contribute where they have data.

**中文**：`mean_sem` 把 `(实验数, 网格点数)` 的堆叠数组压缩为每个网格点的均值、标准误（SEM）和有效样本数 n。它对 NaN 敏感：均值与标准差忽略 NaN，n 只统计该时间点的有限样本，因此部分缺失的实验仍能在其有数据处做出贡献。

In [51]:
# Block 5 — mean ± SEM (cite Step 1 Block 10)

def mean_sem(arr2d):
    if arr2d.size == 0:
        n_cols = arr2d.shape[1] if arr2d.ndim == 2 and arr2d.shape[1] > 0 else 0
        nan = np.full(n_cols, np.nan)
        return nan, nan, np.zeros(n_cols)
    mu = np.nanmean(arr2d, axis=0)
    n_eff = np.sum(np.isfinite(arr2d), axis=0).astype(float)
    sd = np.nanstd(arr2d, axis=0, ddof=1)
    sem = sd / np.sqrt(np.maximum(n_eff, 1.0))
    sem[~np.isfinite(sem)] = np.nan
    return mu, sem, n_eff

### Block 6 — Four-panel mean ± SEM figure

`plot_oscillation_summary` draws one panel per signal: the across-trial mean line with a shaded ± SEM band, using the y-limits from Block 4 (and printing a warning if the band is clipped). Panel 0 is annotated with the trial count `n`. Saves SVG + PNG.

**中文**：`plot_oscillation_summary` 为每个信号画一个子图：跨实验均值曲线加上 ± SEM 的阴影带，y 轴范围沿用 Block 4 的设置（若阴影带被裁切会打印告警）。第一个子图标注实验数 `n`。结果保存为 SVG + PNG。

In [52]:
# Block 6 — 4-panel mean ± SEM figure (uses METRICS + YLIMS from Block 4)

def plot_oscillation_summary(t_grid, stacks, out_svg, out_png,
                             title=None, n_trials=None):
    fig, axes = plt.subplots(len(METRICS), 1, figsize=(14, 12), sharex=True)
    for ax, (col, label, unit) in zip(axes, METRICS):
        arr = stacks[col]
        if arr.shape[0] == 0:
            ax.text(0.5, 0.5, f"no trials for {col}",
                    transform=ax.transAxes, ha="center", va="center")
        else:
            mu, sem, _ = mean_sem(arr)
            ax.plot(t_grid, mu, color="#1f77b4", lw=1.2)
            ax.fill_between(t_grid, mu - sem, mu + sem,
                            color="#1f77b4", alpha=0.25, lw=0)
        ylabel = f"{label} ({unit})" if unit else label
        ax.set_ylabel(ylabel)
        ax.grid(True, linestyle="--", alpha=0.35)
        ylim = YLIMS.get(col)
        if ylim is not None:
            ax.set_ylim(*ylim)
            if arr.shape[0] > 0:
                lo_data = float(np.nanmin(mu - sem))
                hi_data = float(np.nanmax(mu + sem))
                if lo_data < ylim[0] or hi_data > ylim[1]:
                    print(
                        f"[warn] {col}: mean+/-SEM range "
                        f"[{lo_data:.3f}, {hi_data:.3f}] is clipped by "
                        f"y-limits [{ylim[0]}, {ylim[1]}]"
                    )

    # n-trials annotation in upper-left of panel 0
    if n_trials is None:
        n_trials = max((arr.shape[0] for arr in stacks.values()), default=0)
    axes[0].text(
        0.012, 0.88, f"n = {n_trials}",
        transform=axes[0].transAxes, ha="left", va="top", fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                  edgecolor="black", alpha=0.85, linewidth=0.8),
    )

    axes[-1].set_xlabel("Time (s, NIDAQ absolute)")
    axes[-1].set_xlim(t_grid[0], t_grid[-1])
    if title:
        fig.suptitle(title, fontsize=10)
        fig.subplots_adjust(top=0.96)
    plt.tight_layout()
    fig.savefig(out_svg, format="svg", bbox_inches="tight")
    fig.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close(fig)

### Block 7 — Save summary CSV

`save_summary_csv` writes one row per grid point: a `time_s` column plus `<metric>_mean`, `<metric>_sem`, and `<metric>_n` columns for every signal. This is the tabular form of the Block 6 figure.

**中文**：`save_summary_csv` 每个网格点写一行：一列 `time_s`，外加每个信号的 `<metric>_mean`、`<metric>_sem`、`<metric>_n` 三列。这相当于 Block 6 图形的表格版本。

In [53]:
# Block 7 — save summary CSV (time_s + mean/sem/n per metric)

def save_summary_csv(t_grid, stacks, out_csv):
    out = pd.DataFrame({"time_s": t_grid})
    for col, *_ in METRICS:
        arr = stacks[col]
        mu, sem, n = mean_sem(arr)
        if arr.shape[0] == 0:
            mu = np.full(len(t_grid), np.nan)
            sem = np.full(len(t_grid), np.nan)
            n = np.zeros(len(t_grid))
        out[f"{col}_mean"] = mu
        out[f"{col}_sem"] = sem
        out[f"{col}_n"] = n.astype(int)
    out.to_csv(out_csv, index=False)

### Block 8 — Runner (full-window summary)

Ties Blocks 2–7 together: finds the CSVs, stacks them on the `[ABS_T0, ABS_T1]` grid, then writes the summary CSV and the 4-panel figure, printing the stack shapes and output paths.

**中文**：把 Block 2–7 串起来：先找到所有 CSV，在 `[ABS_T0, ABS_T1]` 网格上堆叠，然后写出汇总 CSV 与 4 联图，并打印堆叠数组的形状和输出路径。

In [60]:
# Block 8 — runner

csvs = find_combined_csvs(DATA_DIR, recursive=RECURSIVE)
print(f"Found {len(csvs)} combined CSV(s) in {DATA_DIR}")

t_grid, stacks, trial_labels = stack_trials(csvs, ABS_T0, ABS_T1, N_GRID)
print(f"Stacked {len(trial_labels)} trial(s); grid shape = {t_grid.shape}")
for col, *_ in METRICS:
    arr = stacks[col]
    print(f"  {col}: stack shape = {arr.shape}")

out_csv = DATA_DIR / f"{OUT_PREFIX}.csv"
out_svg = DATA_DIR / f"{OUT_PREFIX}.svg"
out_png = DATA_DIR / f"{OUT_PREFIX}.png"

save_summary_csv(t_grid, stacks, out_csv)
plot_oscillation_summary(
    t_grid, stacks, out_svg, out_png,
    title=f"Oscillation summary: n={len(trial_labels)} trials, t = {ABS_T0:.1f}–{ABS_T1:.1f} s",
    n_trials=len(trial_labels),
)

print(f"saved: {out_csv}")
print(f"saved: {out_svg}")
print(f"saved: {out_png}")

Found 8 combined CSV(s) in C:\Users\Lylah\Desktop\data_processing\oscillating
Stacked 8 trial(s); grid shape = (6000,)
  wingbeat_amplitude: stack shape = (8, 6000)
  wingbeat_frequency: stack shape = (8, 6000)
  inferred_flight_power: stack shape = (8, 6000)
  rescaled_intensity: stack shape = (8, 6000)
saved: C:\Users\Lylah\Desktop\data_processing\oscillating\oscillation_summary_20_80s.csv
saved: C:\Users\Lylah\Desktop\data_processing\oscillating\oscillation_summary_20_80s.svg
saved: C:\Users\Lylah\Desktop\data_processing\oscillating\oscillation_summary_20_80s.png


### Block 9 — Cycle-folded summary

The 20–80 s window contains three repeats of the 20 s oscillation (20–40, 40–60, 60–80 s). `stack_cycles` treats each repeat as a replicate and folds them all onto a single 0–20 s cycle grid; `plot_cycle_summary` then plots mean ± SEM of that folded cycle (same style as Block 6). Reuses `save_summary_csv`, where `time_s` now means time-within-cycle. Outputs `oscillation_cycle_avg_0_20s.*`.

**中文**：20–80 s 窗口包含 20 s 振荡的三个重复段（20–40、40–60、60–80 s）。`stack_cycles` 把每个重复段当作一个重复样本，全部折叠到单个 0–20 s 周期网格上；`plot_cycle_summary` 再画出该折叠周期的 mean ± SEM（样式同 Block 6）。复用 `save_summary_csv`，此时 `time_s` 表示"周期内时间"。输出 `oscillation_cycle_avg_0_20s.*`。

In [63]:
# Block 9 — cycle-folded summary
# The 20–80 s window is three 20-s repeats (20–40, 40–60, 60–80).
# Treat them as replicates and collapse onto a single 0–20 s cycle.

CYCLE_BASES  = [20.0, 40, 60]   # absolute start time of each repeat
CYCLE_LEN    = 20.0                 # seconds per cycle
CYCLE_N_GRID = 2000                 # 100 Hz over 20 s

CYCLE_PREFIX = f"oscillation_cycle_avg_0_{int(CYCLE_LEN)}s"


def stack_cycles(csv_paths, cycle_bases, cycle_len, n_grid):
    """Fold each trial into n_cycles repeats and stack all (n_trials * n_cycles)
    replicates onto one 0..cycle_len grid."""
    t_grid = np.linspace(0.0, cycle_len, n_grid)
    stacks = {col: [] for col, *_ in METRICS}
    n_trials_used = 0
    n_reps_kept = 0
    for p in csv_paths:
        try:
            df = load_combined_csv(p)
        except Exception as e:
            print(f"[warn] {p.name}: {e} — skipping")
            continue
        trial_used = False
        for base in cycle_bases:
            t0, t1 = base, base + cycle_len
            mask = (df["time_s"] >= t0) & (df["time_s"] <= t1)
            sub = df.loc[mask]
            if len(sub) == 0:
                print(f"[warn] {p.name}: no samples in [{t0}, {t1}] s — skipping cycle")
                continue
            t_rel = sub["time_s"].values - base
            for col, *_ in METRICS:
                stacks[col].append(interp_irregular(t_rel, sub[col].values, t_grid))
            n_reps_kept += 1
            trial_used = True
        if trial_used:
            n_trials_used += 1
    stacks = {
        k: (np.asarray(v) if len(v) > 0 else np.empty((0, n_grid)))
        for k, v in stacks.items()
    }
    return t_grid, stacks, n_trials_used, n_reps_kept


def plot_cycle_summary(t_grid, stacks, out_svg, out_png, title=None, cycle_len=CYCLE_LEN, n_trials=None):
    fig, axes = plt.subplots(len(METRICS), 1, figsize=(14, 12), sharex=True)
    for ax, (col, label, unit) in zip(axes, METRICS):
        arr = stacks[col]
        if arr.shape[0] == 0:
            ax.text(0.5, 0.5, f"no data for {col}",
                    transform=ax.transAxes, ha="center", va="center")
        else:
            mu, sem, _ = mean_sem(arr)
            ax.plot(t_grid, mu, color="#1f77b4", lw=1.2)
            ax.fill_between(t_grid, mu - sem, mu + sem,
                            color="#1f77b4", alpha=0.25, lw=0)
        ylabel = f"{label} ({unit})" if unit else label
        ax.set_ylabel(ylabel)
        ax.grid(True, linestyle="--", alpha=0.35)
        ylim = YLIMS.get(col)
        if ylim is not None:
            ax.set_ylim(*ylim)
            if arr.shape[0] > 0:
                lo_data = float(np.nanmin(mu - sem))
                hi_data = float(np.nanmax(mu + sem))
                if lo_data < ylim[0] or hi_data > ylim[1]:
                    print(
                        f"[warn] {col}: mean+/-SEM range "
                        f"[{lo_data:.3f}, {hi_data:.3f}] is clipped by "
                        f"y-limits [{ylim[0]}, {ylim[1]}]"
                    )
    # n-trials annotation in upper-left of panel 0
    if n_trials is None:
        n_trials = max((arr.shape[0] for arr in stacks.values()), default=0)
    axes[0].text(
        0.012, 0.88, f"n = {n_trials}",
        transform=axes[0].transAxes, ha="left", va="top", fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                  edgecolor="black", alpha=0.85, linewidth=0.8),
    )
    axes[-1].set_xlabel("Time in cycle (s)")
    axes[-1].set_xlim(0.0, cycle_len)
    if title:
        fig.suptitle(title, fontsize=10)
        fig.subplots_adjust(top=0.96)
    plt.tight_layout()
    fig.savefig(out_svg, format="svg", bbox_inches="tight")
    fig.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close(fig)


t_grid_c, stacks_c, n_trials_c, n_reps_c = stack_cycles(
    csvs, CYCLE_BASES, CYCLE_LEN, CYCLE_N_GRID
)
print(f"Cycle-fold: {n_trials_c} trial(s) x up to {len(CYCLE_BASES)} cycles = {n_reps_c} replicate(s)")
for col, *_ in METRICS:
    print(f"  {col}: stack shape = {stacks_c[col].shape}")

out_csv_c = DATA_DIR / f"{CYCLE_PREFIX}.csv"
out_svg_c = DATA_DIR / f"{CYCLE_PREFIX}.svg"
out_png_c = DATA_DIR / f"{CYCLE_PREFIX}.png"

# Reuse save_summary_csv: the "time_s" column holds time within the cycle
save_summary_csv(t_grid_c, stacks_c, out_csv_c)
plot_cycle_summary(
    t_grid_c, stacks_c, out_svg_c, out_png_c,
    title=f"Cycle-folded summary: n={n_trials_c} trial(s), {n_reps_c} replicates, 0–{CYCLE_LEN:.0f} s",
    n_trials=n_trials_c,
)

print(f"saved: {out_csv_c}")
print(f"saved: {out_svg_c}")
print(f"saved: {out_png_c}")


Cycle-fold: 8 trial(s) x up to 1 cycles = 8 replicate(s)
  wingbeat_amplitude: stack shape = (8, 2000)
  wingbeat_frequency: stack shape = (8, 2000)
  inferred_flight_power: stack shape = (8, 2000)
  rescaled_intensity: stack shape = (8, 2000)
saved: C:\Users\Lylah\Desktop\data_processing\oscillating\oscillation_cycle_avg_0_20s.csv
saved: C:\Users\Lylah\Desktop\data_processing\oscillating\oscillation_cycle_avg_0_20s.svg
saved: C:\Users\Lylah\Desktop\data_processing\oscillating\oscillation_cycle_avg_0_20s.png


### Block 10 — Per-trial Pearson correlations

For each trial, computes Pearson correlations two ways: (a) spiracle aperture vs each flight signal, and (b) a synthetic 0.05 Hz sine (the visual-stimulus drive) vs every signal. Each pair is scored over three windows — the full `[ABS_T0, ABS_T1]`, plus 20–40 s and 60–80 s — and all values are saved per trial to `pearson_correlations_<T0>_<T1>s.csv`. Also prints across-trial mean ± SD of each correlation. (This CSV is later read by Step3B's Block 11.)

**中文**：对每次实验，分两种方式计算皮尔逊相关：(a) 气门开度 vs 各飞行信号；(b) 合成的 0.05 Hz 正弦波（代表视觉刺激驱动）vs 每个信号。每个信号对都在三个时间窗内计算——完整的 `[ABS_T0, ABS_T1]`，以及 20–40 s 和 60–80 s——所有数值按实验逐行保存到 `pearson_correlations_<T0>_<T1>s.csv`，并打印各相关系数的跨实验均值 ± 标准差。（该 CSV 之后会被 Step3B 的 Block 11 读取。）

In [64]:
# Block 10 — per-trial Pearson correlation: spiracle size vs other signals,
# plus a synthetic 0.05 Hz sine wave (the visual-stimulus drive) vs every signal.
# Each correlation is computed over THREE windows:
#   * full window [ABS_T0, ABS_T1]
#   * 20-40 s     (first repeat of the oscillation)
#   * 60-80 s     (third repeat)
# All three numbers per pair are saved as separate columns in the same CSV.

CORR_TARGET = "rescaled_intensity"  # spiracle size
CORR_PARTNERS = [
    ("wingbeat_amplitude",    "Wingbeat amplitude"),
    ("wingbeat_frequency",    "Wingbeat frequency"),
    ("inferred_flight_power", "Inferred flight power"),
]

# Visual stimulus model: sin(2*pi*f*t) on absolute NIDAQ time. With f = 0.05 Hz
# the zero-crossings fall on t = 20, 40, 60, 80 s, matching the cycle-fold bases.
SINE_FREQ_HZ = 0.05

# Signals correlated against the sine wave (spiracle + the three flight signals).
SINE_PARTNERS = [
    ("rescaled_intensity",    "Spiracle aperture"),
    ("wingbeat_amplitude",    "Wingbeat amplitude"),
    ("wingbeat_frequency",    "Wingbeat frequency"),
    ("inferred_flight_power", "Inferred flight power"),
]

# Per-correlation windows. The first one is the full window; the labels become
# column-name suffixes (`r_<col>_<label>`).
def _corr_windows(abs_t0, abs_t1):
    return [
        (f"{int(abs_t0)}_{int(abs_t1)}s", abs_t0, abs_t1),
        ("20_40s",                        20.0,   40.0),
        ("60_80s",                        60.0,   80.0),
    ]


def _pearson(x, y):
    """Pearson r on finite-pair samples. Returns (r, n_pairs)."""
    m = np.isfinite(x) & np.isfinite(y)
    n = int(m.sum())
    if n < 3:
        return np.nan, n
    xs = x[m]
    ys = y[m]
    if np.std(xs) == 0 or np.std(ys) == 0:
        return np.nan, n
    return float(np.corrcoef(xs, ys)[0, 1]), n


def compute_trial_correlations(csv_paths, abs_t0, abs_t1, sine_freq_hz=SINE_FREQ_HZ):
    windows = _corr_windows(abs_t0, abs_t1)
    rows = []
    for p in csv_paths:
        try:
            df = load_combined_csv(p)
        except Exception as e:
            print(f"[warn] {p.name}: {e} — skipping")
            continue

        row = {"trial": p.stem}
        for label, t0, t1 in windows:
            mask = (df["time_s"] >= t0) & (df["time_s"] <= t1)
            sub = df.loc[mask]
            if len(sub) < 3:
                print(f"[warn] {p.name}: < 3 samples in [{t0}, {t1}] s — filling NaN for {label}")
                for col, _label in CORR_PARTNERS:
                    row[f"r_{col}_{label}"] = np.nan
                    row[f"n_{col}_{label}"] = 0
                for col, _label in SINE_PARTNERS:
                    row[f"r_sine_vs_{col}_{label}"] = np.nan
                    row[f"n_sine_vs_{col}_{label}"] = 0
                continue

            t = sub["time_s"].values.astype(float)
            sine = np.sin(2.0 * np.pi * sine_freq_hz * t)

            x = sub[CORR_TARGET].values.astype(float)
            for col, _label in CORR_PARTNERS:
                r, n = _pearson(x, sub[col].values.astype(float))
                row[f"r_{col}_{label}"] = r
                row[f"n_{col}_{label}"] = n

            for col, _label in SINE_PARTNERS:
                r, n = _pearson(sine, sub[col].values.astype(float))
                row[f"r_sine_vs_{col}_{label}"] = r
                row[f"n_sine_vs_{col}_{label}"] = n

        rows.append(row)
    return pd.DataFrame(rows), windows


corr_df, corr_windows = compute_trial_correlations(csvs, ABS_T0, ABS_T1, SINE_FREQ_HZ)
print(
    f"Computed correlations for {len(corr_df)} trial(s), "
    f"sine = {SINE_FREQ_HZ} Hz, windows = "
    + ", ".join(f"{lab} [{t0:.1f}-{t1:.1f} s]" for lab, t0, t1 in corr_windows)
    + "\n"
)
with pd.option_context("display.max_columns", None, "display.width", 240):
    print(corr_df.to_string(index=False))


def _summarize(label_text, vals):
    if len(vals) == 0:
        print(f"  {label_text}: no data")
    elif len(vals) == 1:
        print(f"  {label_text}: r = {vals[0]:.3f}  (n=1)")
    else:
        print(
            f"  {label_text}: r = {np.mean(vals):.3f} ± {np.std(vals, ddof=1):.3f}  (n={len(vals)})"
        )


for win_label, t0, t1 in corr_windows:
    print(f"\nAcross-trial summary (mean ± std of r) — window {win_label} [{t0:.1f}-{t1:.1f} s]")
    print("  spiracle vs flight signals:")
    for col, label in CORR_PARTNERS:
        _summarize(f"    {label} vs spiracle", corr_df[f"r_{col}_{win_label}"].dropna().values)
    print(f"  {SINE_FREQ_HZ} Hz sine vs every signal:")
    for col, label in SINE_PARTNERS:
        _summarize(f"    {label} vs sine", corr_df[f"r_sine_vs_{col}_{win_label}"].dropna().values)

out_corr_csv = DATA_DIR / f"pearson_correlations_{int(ABS_T0)}_{int(ABS_T1)}s.csv"
corr_df.to_csv(out_corr_csv, index=False)
print(f"\nsaved: {out_corr_csv}")

Computed correlations for 8 trial(s), sine = 0.05 Hz, windows = 20_80s [20.0-80.0 s], 20_40s [20.0-40.0 s], 60_80s [60.0-80.0 s]

                                                                                                    trial  r_wingbeat_amplitude_20_80s  n_wingbeat_amplitude_20_80s  r_wingbeat_frequency_20_80s  n_wingbeat_frequency_20_80s  r_inferred_flight_power_20_80s  n_inferred_flight_power_20_80s  r_sine_vs_rescaled_intensity_20_80s  n_sine_vs_rescaled_intensity_20_80s  r_sine_vs_wingbeat_amplitude_20_80s  n_sine_vs_wingbeat_amplitude_20_80s  r_sine_vs_wingbeat_frequency_20_80s  n_sine_vs_wingbeat_frequency_20_80s  r_sine_vs_inferred_flight_power_20_80s  n_sine_vs_inferred_flight_power_20_80s  r_wingbeat_amplitude_20_40s  n_wingbeat_amplitude_20_40s  r_wingbeat_frequency_20_40s  n_wingbeat_frequency_20_40s  r_inferred_flight_power_20_40s  n_inferred_flight_power_20_40s  r_sine_vs_rescaled_intensity_20_40s  n_sine_vs_rescaled_intensity_20_40s  r_sine_vs_wingbeat_amplitud

### Block 10b — Paired comparison of correlation scores (20–40 s vs 60–80 s)

Tests whether each correlation changes between the first oscillation repeat (20–40 s) and the third (60–80 s), pairing within trial. For each channel it runs a paired t-test and a Wilcoxon signed-rank test, reports mean difference, 95% CI and Cohen's dz, and draws a box + jittered scatter + paired connecting-line panel. Uses editable-text SVG settings. Saves the figure (SVG/PDF) and `pearson_corr_20_40s_vs_60_80s_paired_stats.csv`.

**中文**：检验每个相关系数在第一个振荡重复段（20–40 s）与第三个（60–80 s）之间是否变化，按实验做配对。对每个通道运行配对 t 检验和 Wilcoxon 符号秩检验，报告均值差、95% 置信区间和 Cohen's dz，并绘制箱线图 + 抖动散点 + 配对连线图。使用可编辑文字的 SVG 设置。保存图（SVG/PDF）及 `pearson_corr_20_40s_vs_60_80s_paired_stats.csv`。

In [65]:
# Block 10b — paired comparison of correlation scores: 20-40 s vs 60-80 s
# Per pair: paired t-test + Wilcoxon signed-rank + mean +/- SEM + 95% CI + Cohen's dz.
# Per pair: box + jittered scatter + paired connecting lines.
# Inputs come from `corr_df` produced by Block 10.

from scipy import stats as _stats
import matplotlib as _mpl

W1_LABEL = "20_40s"
W2_LABEL = "60_80s"

# All correlation channels computed in Block 10. Each entry: (column stem, pretty label)
_CORR_CHANNELS = (
    [(f"r_{col}",            f"{label} vs spiracle")  for col, label in CORR_PARTNERS]
    + [(f"r_sine_vs_{col}",  f"{label} vs sine")      for col, label in SINE_PARTNERS]
)


def _set_editable_svg_style():
    """Editable-text SVGs (Affinity / Illustrator safe). Idempotent."""
    _mpl.rcParams["font.family"]      = "sans-serif"
    _mpl.rcParams["font.sans-serif"]  = ["Arial"]
    _mpl.rcParams["svg.fonttype"]     = "none"
    _mpl.rcParams["pdf.fonttype"]     = 42
    _mpl.rcParams["ps.fonttype"]      = 42
    _mpl.rcParams["figure.dpi"]       = 300
    _mpl.rcParams["savefig.dpi"]      = 300


def _paired_stats(x, y, label):
    """Paired t + Wilcoxon for x (20-40 s) vs y (60-80 s). diff = y - x."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    n = len(x)
    diff = y - x

    if n >= 2:
        t_stat, t_p = _stats.ttest_rel(y, x)
    else:
        t_stat, t_p = (np.nan, np.nan)

    if n >= 1 and np.any(diff != 0):
        try:
            method = "exact" if n <= 25 else "auto"
            w_stat, w_p = _stats.wilcoxon(y, x, zero_method="wilcox",
                                          alternative="two-sided", method=method)
        except ValueError:
            w_stat, w_p = (np.nan, np.nan)
    else:
        w_stat, w_p = (np.nan, np.nan)

    mean_diff = float(np.mean(diff)) if n else np.nan
    sd_diff   = float(np.std(diff, ddof=1)) if n > 1 else np.nan
    sem_diff  = sd_diff / np.sqrt(n) if n > 1 else np.nan
    tcrit     = _stats.t.ppf(0.975, df=n - 1) if n > 1 else np.nan
    ci_lo     = mean_diff - tcrit * sem_diff if n > 1 else np.nan
    ci_hi     = mean_diff + tcrit * sem_diff if n > 1 else np.nan
    cohen_dz  = mean_diff / sd_diff if (sd_diff and sd_diff > 0) else np.nan

    return {
        "comparison":      label,
        "n":               n,
        "mean_w1":         float(np.mean(x))                    if n else np.nan,
        "sem_w1":          float(np.std(x, ddof=1) / np.sqrt(n)) if n > 1 else np.nan,
        "mean_w2":         float(np.mean(y))                    if n else np.nan,
        "sem_w2":          float(np.std(y, ddof=1) / np.sqrt(n)) if n > 1 else np.nan,
        "mean_diff":       mean_diff,
        "sem_diff":        sem_diff,
        "ci95_low":        ci_lo,
        "ci95_high":       ci_hi,
        "cohen_dz":        cohen_dz,
        "t_stat":          t_stat,
        "t_pvalue":        t_p,
        "wilcoxon_stat":   w_stat,
        "wilcoxon_pvalue": w_p,
    }


def _stars(p):
    if p is None or not np.isfinite(p): return "n.s."
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "n.s."


_PALETTE = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd",
            "#ff7f0e", "#17becf", "#8c564b", "#e377c2"]


def plot_corr_window_comparison(corr_df, channels, w1_label, w2_label,
                                out_svg, out_pdf, ncols=4):
    nplots = len(channels)
    nrows  = int(np.ceil(nplots / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.5 * ncols, 3.6 * nrows),
                             squeeze=False)
    axes_flat = axes.flatten()
    results = {}

    n_for_title = 0
    rng = np.random.default_rng(0)

    for i, (stem, label) in enumerate(channels):
        ax = axes_flat[i]
        col1 = f"{stem}_{w1_label}"
        col2 = f"{stem}_{w2_label}"
        if col1 not in corr_df.columns or col2 not in corr_df.columns:
            ax.text(0.5, 0.5, f"missing\n{col1}\nor\n{col2}",
                    transform=ax.transAxes, ha="center", va="center", fontsize=8)
            ax.set_axis_off()
            continue

        x = corr_df[col1].to_numpy(dtype=float)
        y = corr_df[col2].to_numpy(dtype=float)
        m = np.isfinite(x) & np.isfinite(y)
        x_v, y_v = x[m], y[m]

        res = _paired_stats(x_v, y_v, label)
        results[stem] = res
        n_for_title = max(n_for_title, res["n"])

        color = _PALETTE[i % len(_PALETTE)]
        positions = np.array([0, 1])
        bp = ax.boxplot(
            [x_v, y_v], positions=positions, widths=0.55,
            showfliers=False, patch_artist=True,
            medianprops=dict(color="black", linewidth=1.2),
        )
        for patch in bp["boxes"]:
            patch.set_facecolor(color)
            patch.set_alpha(0.35)
            patch.set_edgecolor("black")
            patch.set_linewidth(0.8)
        for element in ("whiskers", "caps"):
            for line in bp[element]:
                line.set_color("black")
                line.set_linewidth(0.8)

        xs_a = positions[0] + rng.uniform(-0.08, 0.08, size=len(x_v))
        xs_b = positions[1] + rng.uniform(-0.08, 0.08, size=len(y_v))
        ax.scatter(xs_a, x_v, color=color, edgecolor="black",
                   linewidth=0.5, s=32, zorder=3)
        ax.scatter(xs_b, y_v, color=color, edgecolor="black",
                   linewidth=0.5, s=32, zorder=3)
        for xa, ya, xb, yb in zip(xs_a, x_v, xs_b, y_v):
            ax.plot([xa, xb], [ya, yb], color="gray", alpha=0.4, lw=0.7, zorder=2)

        p_w, p_t = res["wilcoxon_pvalue"], res["t_pvalue"]
        pw_str = f"p_w = {p_w:.3g} {_stars(p_w)}" if np.isfinite(p_w) else "p_w = n/a"
        pt_str = f"p_t = {p_t:.3g} {_stars(p_t)}" if np.isfinite(p_t) else "p_t = n/a"
        ax.text(0.02, 0.98, f"{pw_str}\n{pt_str}",
                transform=ax.transAxes, va="top", ha="left", fontsize=7.5,
                family="monospace",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                          alpha=0.85, edgecolor="lightgray"))

        ax.set_xticks(positions)
        ax.set_xticklabels([w1_label.replace("_", "-"),
                            w2_label.replace("_", "-")])
        ax.set_xlim(-0.6, 1.6)
        ax.set_ylabel("Pearson r")
        ax.set_title(label, fontsize=9)
        ax.axhline(0.0, color="black", lw=0.6, ls=":", alpha=0.6)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ymin, ymax = ax.get_ylim()
        ax.set_ylim(ymin, ymax + 0.12 * (ymax - ymin))

    # hide any leftover axes
    for j in range(len(channels), len(axes_flat)):
        axes_flat[j].set_axis_off()

    fig.suptitle(f"Paired correlation comparison: {w1_label} vs {w2_label}  "
                 f"(n = {n_for_title})", fontsize=10, y=1.00)
    fig.tight_layout()
    fig.savefig(out_svg, format="svg", bbox_inches="tight")
    fig.savefig(out_pdf, format="pdf", bbox_inches="tight")
    plt.close(fig)
    return results


_set_editable_svg_style()

out_cmp_svg = DATA_DIR / f"pearson_corr_{W1_LABEL}_vs_{W2_LABEL}.svg"
out_cmp_pdf = DATA_DIR / f"pearson_corr_{W1_LABEL}_vs_{W2_LABEL}.pdf"
out_cmp_csv = DATA_DIR / f"pearson_corr_{W1_LABEL}_vs_{W2_LABEL}_paired_stats.csv"

cmp_results = plot_corr_window_comparison(
    corr_df, _CORR_CHANNELS, W1_LABEL, W2_LABEL,
    out_cmp_svg, out_cmp_pdf, ncols=4,
)

# Console summary + save stats CSV
print(f"\nPaired comparison of correlation scores: {W1_LABEL} vs {W2_LABEL}")
print(f"{'channel':<40s} {'n':>3s}  {'mean diff':>10s}  {'95% CI':>22s}  "
      f"{'dz':>6s}  {'t (p)':>16s}  {'W (p)':>16s}")
print("-" * 130)
rows_out = []
for stem, label in _CORR_CHANNELS:
    r = cmp_results.get(stem)
    if r is None:
        continue
    ci = (f"[{r['ci95_low']:+.3f}, {r['ci95_high']:+.3f}]"
          if np.isfinite(r["ci95_low"]) else "n/a")
    t_str = (f"{r['t_stat']:+.2f} ({r['t_pvalue']:.3g})"
             if np.isfinite(r["t_pvalue"]) else "n/a")
    w_str = (f"{r['wilcoxon_stat']:.1f} ({r['wilcoxon_pvalue']:.3g})"
             if np.isfinite(r["wilcoxon_pvalue"]) else "n/a")
    dz    = f"{r['cohen_dz']:+.2f}" if np.isfinite(r["cohen_dz"]) else "n/a"
    md    = f"{r['mean_diff']:+.3f}" if np.isfinite(r["mean_diff"]) else "n/a"
    print(f"{label:<40s} {r['n']:>3d}  {md:>10s}  {ci:>22s}  "
          f"{dz:>6s}  {t_str:>16s}  {w_str:>16s}")
    row = {"channel_stem": stem, **r}
    rows_out.append(row)

pd.DataFrame(rows_out).to_csv(out_cmp_csv, index=False)
print(f"\nsaved: {out_cmp_csv}")
print(f"saved: {out_cmp_svg}")
print(f"saved: {out_cmp_pdf}")


Paired comparison of correlation scores: 20_40s vs 60_80s
channel                                    n   mean diff                  95% CI      dz             t (p)             W (p)
----------------------------------------------------------------------------------------------------------------------------------
Wingbeat amplitude vs spiracle             8      -0.011        [-0.404, +0.383]   -0.02      -0.06 (0.95)          18.0 (1)
Wingbeat frequency vs spiracle             8      +0.005        [-0.490, +0.501]   +0.01      +0.03 (0.98)      16.0 (0.844)
Inferred flight power vs spiracle          8      -0.060        [-0.322, +0.201]   -0.19     -0.55 (0.602)      16.0 (0.844)
Spiracle aperture vs sine                  8      -0.014        [-0.320, +0.292]   -0.04     -0.11 (0.917)      17.0 (0.945)
Wingbeat amplitude vs sine                 8      -0.107        [-0.532, +0.317]   -0.21     -0.60 (0.569)      14.0 (0.641)
Wingbeat frequency vs sine                 8      +0.087    

### Block 11 — Spiracle peak / trough / amplitude comparison (20–40 s vs 60–80 s)

For each trial and each window, summarizes the spiracle aperture using a top/bottom-10% method: the mean of the top 10% of samples (peak), the mean of the bottom 10% (trough), and their difference (swing/amplitude). Each metric is paired across the two windows within a trial and tested with both Wilcoxon signed-rank and paired t-tests. Saves a single CSV (`spiracle_peak_trough_comparison_20_40_vs_60_80.csv`) holding per-trial rows plus MEAN/SEM and test-statistic rows.

**中文**：对每次实验、每个时间窗，用"前/后 10%"方法概括气门开度：取最高 10% 样本的均值（峰）、最低 10% 样本的均值（谷）、以及二者之差（摆幅/振幅）。每个指标在同一实验的两个窗之间配对，并同时用 Wilcoxon 符号秩检验和配对 t 检验。结果保存为单个 CSV（`spiracle_peak_trough_comparison_20_40_vs_60_80.csv`），含逐实验行以及 MEAN/SEM 和检验统计量行。

In [ ]:
# Block 11 — peak / trough / amplitude comparison for spiracle aperture
# Window 1 (20-40 s) vs Window 2 (60-80 s). Top/bottom-10% method only.
# Each metric is paired across windows within a trial; we report Wilcoxon
# signed-rank AND paired-t test for the (w1 - w2) within-trial difference.

from scipy.stats import wilcoxon, ttest_rel

WINDOWS_PT       = [(20.0, 40.0), (60.0, 80.0)]
PERCENT_LO       = 10.0   # bottom 10 %
PERCENT_HI       = 90.0   # top 10 %
MIN_SAMPLES      = 3      # per subset (top-10% / bottom-10%)
MIN_PAIRS_FOR_PT = 5      # below this, skip the paired test

METRIC_TYPES = [
    "top10_mean",
    "bottom10_mean",
    "top10_minus_bottom10",
]


def compute_window_metrics(df, abs_t0, abs_t1):
    """Three numbers for spiracle aperture in [abs_t0, abs_t1] of one trial."""
    out = {m: np.nan for m in METRIC_TYPES}
    mask = (df["time_s"] >= abs_t0) & (df["time_s"] <= abs_t1)
    sub = df.loc[mask]
    if len(sub) == 0:
        return out
    y = pd.to_numeric(sub["rescaled_intensity"], errors="coerce").values.astype(float)
    valid_y = np.isfinite(y)
    if valid_y.sum() < MIN_SAMPLES:
        return out

    p_lo = float(np.nanpercentile(y[valid_y], PERCENT_LO))
    p_hi = float(np.nanpercentile(y[valid_y], PERCENT_HI))
    top_mask = valid_y & (y >= p_hi)
    bot_mask = valid_y & (y <= p_lo)
    if top_mask.sum() >= MIN_SAMPLES:
        out["top10_mean"] = float(np.nanmean(y[top_mask]))
    if bot_mask.sum() >= MIN_SAMPLES:
        out["bottom10_mean"] = float(np.nanmean(y[bot_mask]))
    if np.isfinite(out["top10_mean"]) and np.isfinite(out["bottom10_mean"]):
        out["top10_minus_bottom10"] = out["top10_mean"] - out["bottom10_mean"]
    return out


def paired_test(x_arr, y_arr, fn):
    """Paired test, dropping pairs with any non-finite element. Returns (stat, p, n_pairs)."""
    x = np.asarray(x_arr, dtype=float)
    y = np.asarray(y_arr, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    n = int(m.sum())
    if n < MIN_PAIRS_FOR_PT:
        return np.nan, np.nan, n
    try:
        res = fn(x[m], y[m])
        return float(res.statistic), float(res.pvalue), n
    except Exception as e:
        print(f"  [warn] paired test failed: {type(e).__name__}: {e}")
        return np.nan, np.nan, n


# ---- Build per-trial rows ----
csvs_pt = find_combined_csvs(DATA_DIR, recursive=RECURSIVE)
print(f"Found {len(csvs_pt)} combined CSV(s) under {DATA_DIR}")

rows = []
for p in csvs_pt:
    try:
        df_pt = load_combined_csv(p)
    except Exception as e:
        print(f"  [warn] {p.name}: {e} -- skipping")
        continue
    m1 = compute_window_metrics(df_pt, *WINDOWS_PT[0])
    m2 = compute_window_metrics(df_pt, *WINDOWS_PT[1])
    row = {"trial": p.stem.replace("_combined", "")}
    for k, v in m1.items():
        row[f"20-40s_{k}"] = v
    for k, v in m2.items():
        row[f"60-80s_{k}"] = v
    rows.append(row)
    print(
        f"  {row['trial']}  "
        f"top10 w1/w2 = {m1['top10_mean']:.3f} / {m2['top10_mean']:.3f}   "
        f"swing w1/w2 = {m1['top10_minus_bottom10']:.3f} / {m2['top10_minus_bottom10']:.3f}"
    )

if not rows:
    raise RuntimeError("No trials processed -- check DATA_DIR.")

trial_df = pd.DataFrame(rows)
metric_cols_w1 = [f"20-40s_{m}" for m in METRIC_TYPES]
metric_cols_w2 = [f"60-80s_{m}" for m in METRIC_TYPES]
ordered_cols = ["trial"] + metric_cols_w1 + metric_cols_w2
trial_df = trial_df.reindex(columns=ordered_cols)


# ---- MEAN, SEM rows ----
def _col_mean_sem(col_values):
    arr = np.asarray(col_values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan, np.nan
    if len(arr) == 1:
        return float(arr[0]), np.nan
    return float(arr.mean()), float(arr.std(ddof=1) / np.sqrt(len(arr)))


mean_row = {c: np.nan for c in ordered_cols}
sem_row  = {c: np.nan for c in ordered_cols}
mean_row["trial"] = "MEAN"
sem_row["trial"]  = "SEM"
for c in metric_cols_w1 + metric_cols_w2:
    mu, se = _col_mean_sem(trial_df[c].values)
    mean_row[c] = mu
    sem_row[c]  = se


# ---- Paired tests: 3 metrics x 2 tests = 6 stat rows ----
# Each stat row has the test statistic in the w1 column and the p-value in the
# w2 column for that metric type. The test compares (x_w1, x_w2) PAIRED.
stat_rows = []
console_lines = []
for metric in METRIC_TYPES:
    col_w1 = f"20-40s_{metric}"
    col_w2 = f"60-80s_{metric}"
    x = trial_df[col_w1].values
    y = trial_df[col_w2].values

    w_stat, w_p, n_pairs = paired_test(x, y, wilcoxon)
    t_stat, t_p, _       = paired_test(x, y, ttest_rel)

    row_w = {c: np.nan for c in ordered_cols}
    row_w["trial"] = f"Wilcoxon ({metric}, n={n_pairs})"
    row_w[col_w1] = w_stat
    row_w[col_w2] = w_p
    stat_rows.append(row_w)

    row_t = {c: np.nan for c in ordered_cols}
    row_t["trial"] = f"Paired t ({metric}, n={n_pairs})"
    row_t[col_w1] = t_stat
    row_t[col_w2] = t_p
    stat_rows.append(row_t)

    console_lines.append((metric, w_stat, w_p, t_stat, t_p, n_pairs))


# ---- Assemble + save ----
footer_df = pd.DataFrame([mean_row, sem_row] + stat_rows)
out_df = pd.concat([trial_df, footer_df], ignore_index=True)
out_df = out_df.reindex(columns=ordered_cols)

OUT_PT_CSV = DATA_DIR / "spiracle_peak_trough_comparison_20_40_vs_60_80.csv"
out_df.to_csv(OUT_PT_CSV, index=False)
print(
    f"\nSaved: {OUT_PT_CSV}"
    f"\n  {len(trial_df)} trial rows + 2 summary rows + {len(stat_rows)} stat rows = {len(out_df)} rows total"
)

# ---- Console summary ----
print("\n==== Paired comparison: 20-40 s vs 60-80 s (spiracle aperture, top/bottom-10% method) ====")
print(f"{'metric':<28s} {'Wilcoxon (W, p)':<32s} {'Paired-t (t, p)':<30s} {'n':>5s}")
print("-" * 100)
for metric, w_stat, w_p, t_stat, t_p, n_pairs in console_lines:
    if np.isfinite(w_p):
        wcol = f"W={w_stat:8.3f}, p={w_p:.4g}"
    else:
        wcol = "(insufficient)"
    if np.isfinite(t_p):
        tcol = f"t={t_stat:7.3f}, p={t_p:.4g}"
    else:
        tcol = "(insufficient)"
    print(f"{metric:<28s} {wcol:<32s} {tcol:<30s} {n_pairs:>5d}")


Found 8 combined CSV(s) under C:\Users\Lylah\Desktop\data_processing\oscillating
  2026_0426_102801_empty_ChR_empty_ChR_4d_F_Fly1_Trial3_0ms_thorax_sp1_oscillating_sine_wave_20_ON  top10 w1/w2 = 0.998 / 0.752   swing w1/w2 = 0.892 / 0.548
  2026_0426_115929_empty_ChR_empty_ChR_4d_F_Fly4_Trial2_0ms_thorax_sp1_oscillating_sine_wave_20_ON  top10 w1/w2 = 0.766 / 0.817   swing w1/w2 = 0.673 / 0.765
  2026_0427_144337_empty_ChR_empty_ChR_4d_F_Fly1_Trial2_0ms_thorax_sp1_oscillating_sine_wave_20_ON  top10 w1/w2 = 0.958 / 0.942   swing w1/w2 = 0.883 / 0.874
  2026_0427_150532_empty_ChR_empty_ChR_4d_F_Fly3_Trial1_0ms_thorax_sp1_oscillating_sine_wave_20_ON  top10 w1/w2 = 0.964 / 0.724   swing w1/w2 = 0.880 / 0.628
  2026_0427_154349_empty_ChR_empty_ChR_4d_F_Fly4_Trial1_0ms_thorax_sp1_oscillating_sine_wave_20_ON  top10 w1/w2 = 0.916 / 0.923   swing w1/w2 = 0.878 / 0.834
  2026_0427_160328_empty_ChR_empty_ChR_4d_F_Fly5_Trial2_0ms_thorax_sp1_oscillating_sine_wave_20_ON  top10 w1/w2 = 1.000 / 0.829  